## Goal:

Build simple benchmark models to predict `subrogation` (0/1), save the trained model artifacts, and append each model's validation performance to a single JSON experiment log.

This notebook starts with baseline Logistic Regression and XGBoost models. The JSON log will allow us to compare future fine-tuning runs against these baselines.

## 1. Imports and project paths

In [10]:
import json
import joblib
import pandas as pd

from pathlib import Path
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


In [11]:
# If this notebook is inside a notebooks/ folder, this points to the project root.
# Otherwise, it uses the current working directory as the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "train_features.csv"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
METRICS_PATH = REPORTS_DIR / "model_metrics.json"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Metrics path:", METRICS_PATH)

Project root: /Users/eugene/Desktop/Emory/Projects/subrogation-project
Data path: /Users/eugene/Desktop/Emory/Projects/subrogation-project/data/processed/train_features.csv
Metrics path: /Users/eugene/Desktop/Emory/Projects/subrogation-project/reports/model_metrics.json


## 2. Load processed training data

In [12]:
df = pd.read_csv(DATA_PATH)
df.head()

,subrogation,email_or_tel_available,safety_rating,high_education_ind,address_change_ind,accident_site,witness_present_ind,liab_prct,channel,policy_report_filed_ind,...,high_value_low_liability_ind,high_value_strong_docs_ind,payout_vehicle_ratio_band,vehicle_value_band,vehicle_age_at_claim,mileage_per_vehicle_year,in_network_bodyshop_flag,in_network_high_value_claim_ind,weekend_claim_ind,accident_type_site
0,1,0,75.0,1,1,Parking Area,1,31.0,Broker,1,...,1,1,low,"(14999.999, 19639.785]",0.0,75421.0,0,0,1,multi_vehicle_clear_Parking Area
1,0,1,94.0,1,1,Parking Area,0,34.0,Phone,1,...,0,0,very_low,"(19639.785, 42609.417]",0.0,31988.0,1,0,0,multi_vehicle_clear_Parking Area
2,0,1,76.0,1,1,Unknown,0,39.0,Broker,1,...,1,1,low,"(14999.999, 19639.785]",0.0,60876.0,1,1,0,multi_vehicle_unclear_Unknown
3,1,1,54.0,1,1,Unknown,0,32.0,Phone,1,...,0,0,very_low,"(14999.999, 19639.785]",0.0,152772.0,1,0,1,multi_vehicle_unclear_Unknown
4,0,1,54.0,1,1,Highway/Intersection,0,28.0,Online,0,...,1,0,low,"(42609.417, 130000.0]",0.0,41151.0,1,1,1,multi_vehicle_clear_Highway/Intersection


## 3. Train/validation split and encoding

For this baseline notebook, we use a simple and reproducible preprocessing approach:

1. Separate features and target.
2. Drop identifier columns that should not be used as model predictors.
3. Split the data into training and validation sets.
4. One-hot encode categorical variables.
5. Align validation columns to the training columns.
6. Clean encoded feature names so they are compatible with XGBoost.

This keeps the modeling notebook readable while ensuring that newly engineered categorical features are actually used.


In [13]:
TARGET_COL = "subrogation"
ID_COLS = ["claim_number"]

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Drop identifier columns if present. These identify rows but should not be predictors.
X = X.drop(columns=[col for col in ID_COLS if col in X.columns])

print("Feature matrix shape before encoding:", X.shape)
print("Target distribution:")
print(y.value_counts(normalize=True))

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Categorical columns to encode:")
print(categorical_cols)

print("Numeric columns:")
print(numeric_cols)


def clean_xgboost_feature_names(columns):
    """
    XGBoost does not allow feature names containing [, ], or <.
    pd.cut/pd.qcut category labels can create those characters after one-hot encoding.
    This helper makes feature names safe while keeping them readable.
    """
    return (
        pd.Index(columns)
        .astype(str)
        .str.replace("[", "(", regex=False)
        .str.replace("]", ")", regex=False)
        .str.replace("<", "less_than", regex=False)
    )


Feature matrix shape before encoding: (17999, 41)
Target distribution:
subrogation
0    0.771376
1    0.228624
Name: proportion, dtype: float64
Categorical columns to encode:
['accident_site', 'channel', 'accident_type', 'in_network_bodyshop', 'liability_band', 'claim_amount_band', 'payout_vehicle_ratio_band', 'vehicle_value_band', 'accident_type_site']
Numeric columns:
['email_or_tel_available', 'safety_rating', 'high_education_ind', 'address_change_ind', 'witness_present_ind', 'liab_prct', 'policy_report_filed_ind', 'vehicle_made_year', 'vehicle_weight', 'vehicle_mileage', 'claim_quarter', 'is_holiday_season', 'log_claim_est_payout', 'log_vehicle_price', 'log_annual_income', 'low_policyholder_liability_ind', 'high_policyholder_liability_ind', 'multi_vehicle_ind', 'clear_liability_accident_ind', 'single_car_ind', 'documentation_strength_score', 'strong_documentation_ind', 'low_liability_strong_docs_ind', 'clear_multi_vehicle_with_report_ind', 'high_claim_amount_ind', 'high_value_low_l

In [14]:
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# One-hot encode categorical variables after the split.
X_train = pd.get_dummies(X_train_raw, drop_first=True, dtype=int)
X_val = pd.get_dummies(X_val_raw, drop_first=True, dtype=int)

# Align validation columns to match training columns.
# Any category that appears in validation but not training is dropped.
# Any category that appears in training but not validation is filled with 0.
X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=0)

# Clean feature names for XGBoost compatibility.
X_train.columns = clean_xgboost_feature_names(X_train.columns)
X_val.columns = clean_xgboost_feature_names(X_val.columns)

feature_columns = X_train.columns.tolist()

# Quality checks before modeling.
remaining_categorical_train = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
remaining_categorical_val = X_val.select_dtypes(include=["object", "category"]).columns.tolist()
bad_xgb_columns = [
    col for col in X_train.columns
    if any(char in str(col) for char in ["[", "]", "<"])
]

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Remaining categorical columns in training:", remaining_categorical_train)
print("Remaining categorical columns in validation:", remaining_categorical_val)
print("XGBoost-incompatible column names:", bad_xgb_columns)
print("Target distribution in training:")
print(y_train.value_counts(normalize=True))


Training shape: (14399, 63)
Validation shape: (3600, 63)
Remaining categorical columns in training: []
Remaining categorical columns in validation: []
XGBoost-incompatible column names: []
Target distribution in training:
subrogation
0    0.771373
1    0.228627
Name: proportion, dtype: float64


In [15]:
# Preview encoded feature names.
feature_columns[:30]


['email_or_tel_available',
 'safety_rating',
 'high_education_ind',
 'address_change_ind',
 'witness_present_ind',
 'liab_prct',
 'policy_report_filed_ind',
 'vehicle_made_year',
 'vehicle_weight',
 'vehicle_mileage',
 'claim_quarter',
 'is_holiday_season',
 'log_claim_est_payout',
 'log_vehicle_price',
 'log_annual_income',
 'low_policyholder_liability_ind',
 'high_policyholder_liability_ind',
 'multi_vehicle_ind',
 'clear_liability_accident_ind',
 'single_car_ind',
 'documentation_strength_score',
 'strong_documentation_ind',
 'low_liability_strong_docs_ind',
 'clear_multi_vehicle_with_report_ind',
 'high_claim_amount_ind',
 'high_value_low_liability_ind',
 'high_value_strong_docs_ind',
 'vehicle_age_at_claim',
 'mileage_per_vehicle_year',
 'in_network_bodyshop_flag']

## 4. Reusable evaluation and experiment logging functions

These functions:
- evaluate a fitted model,
- save the model as a `.pkl` file,
- append model metrics to one JSON file,
- store run metadata such as timestamp, parameters, model path, and feature columns.


In [16]:
def make_json_safe(value):
    """
    Convert values that may not be JSON serializable into safe Python objects.
    """
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)


def evaluate_model(model, X_val, y_val):
    """
    Evaluate a fitted binary classification model on validation data.
    Return only the key metrics needed for experiment tracking.
    """
    y_pred = model.predict(X_val)

    metrics = {
        "accuracy": round(accuracy_score(y_val, y_pred), 4),
        "precision": round(precision_score(y_val, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_val, y_pred, zero_division=0), 4),
        "f1_score": round(f1_score(y_val, y_pred, zero_division=0), 4),
    }

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_val)[:, 1]
        metrics["roc_auc"] = round(roc_auc_score(y_val, y_proba), 4)
    else:
        metrics["roc_auc"] = None

    return metrics


def save_model_and_append_metrics(
    model,
    model_name,
    model_family,
    X_val,
    y_val,
    metrics_path=METRICS_PATH,
    models_dir=MODELS_DIR,
    notes=None
):
    """
    Save a trained model and append key validation metrics to a single JSON file.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    run_id = f"{timestamp}_{model_name}"

    model_path = models_dir / f"{run_id}.pkl"

    # Save trained model artifact
    joblib.dump(model, model_path)

    # Evaluate model
    metrics = evaluate_model(model, X_val, y_val)

    # Keep the experiment log simple and readable
    run_record = {
        "run_id": run_id,
        "run_date": datetime.now().isoformat(timespec="seconds"),
        "model_name": model_name,
        "model_family": model_family,
        "model_path": str(model_path),
        "metrics": metrics,
        "notes": notes
    }

    # Load existing experiment log if it exists
    if metrics_path.exists():
        with open(metrics_path, "r") as file:
            records = json.load(file)

        # Handles the case where an older metrics file was stored as a dict
        if isinstance(records, dict):
            records = [records]
    else:
        records = []

    records.append(run_record)

    with open(metrics_path, "w") as file:
        json.dump(records, file, indent=4)

    print(f"Saved model: {model_path}")
    print(f"Appended metrics to: {metrics_path}")

    return run_record


## 5. Baseline Model 1: Logistic Regression

This is the main interpretable benchmark. Because subrogation is likely imbalanced, we use `class_weight="balanced"` and track precision, recall, F1, and ROC-AUC.



In [17]:
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

log_reg.fit(X_train, y_train)

log_reg_record = save_model_and_append_metrics(
    model=log_reg,
    model_name="logistic_regression_baseline_v2",
    model_family="LogisticRegression",
    X_val=X_val,
    y_val=y_val,
    notes="Baseline logistic regression using engineered features and one-hot encoded categorical variables."
)

log_reg_record["metrics"]


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Saved model: /Users/eugene/Desktop/Emory/Projects/subrogation-project/models/2026-04-28_230732_logistic_regression_baseline_v2.pkl
Appended metrics to: /Users/eugene/Desktop/Emory/Projects/subrogation-project/reports/model_metrics.json


{'accuracy': 0.7703,
 'precision': 0.4981,
 'recall': 0.6403,
 'f1_score': 0.5603,
 'roc_auc': 0.8162}

## 6. Baseline Model 2: XGBoost

This provides a stronger tree-based baseline that can capture non-linear patterns.


In [18]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_record = save_model_and_append_metrics(
    model=xgb,
    model_name="xgboost_baseline_v2",
    model_family="XGBClassifier",
    X_val=X_val,
    y_val=y_val,
    notes="Baseline XGBoost using engineered features, encoded categorical variables, and cleaned feature names."
)

xgb_record["metrics"]


Saved model: /Users/eugene/Desktop/Emory/Projects/subrogation-project/models/2026-04-28_230734_xgboost_baseline_v2.pkl
Appended metrics to: /Users/eugene/Desktop/Emory/Projects/subrogation-project/reports/model_metrics.json


{'accuracy': 0.8078,
 'precision': 0.618,
 'recall': 0.4168,
 'f1_score': 0.4978,
 'roc_auc': 0.8179}

## 7. Compare logged model runs

This reads the JSON experiment log and converts it into a table for easier comparison.


In [19]:
with open(METRICS_PATH, "r") as file:
    records = json.load(file)

metrics_df = pd.json_normalize(records)

comparison_cols = [
    "run_id",
    "model_name",
    "model_family",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1_score",
    "metrics.roc_auc",
    "model_path",
    "notes"
]

metrics_df[comparison_cols].sort_values(
    by="metrics.f1_score",
    ascending=False
).head(20)


,run_id,model_name,model_family,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,model_path,notes
0,2026-04-27_140727_logistic_regression_baseline_v1,logistic_regression_baseline_v1,LogisticRegression,0.7344,0.4511,0.7448,0.5619,0.8144,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline logistic regression with class_weight...
2,2026-04-27_140757_logistic_regression_baseline_v1,logistic_regression_baseline_v1,LogisticRegression,0.7344,0.4511,0.7448,0.5619,0.8144,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline logistic regression with class_weight...
4,2026-04-28_061722_logistic_regression_baseline_v1,logistic_regression_baseline_v1,LogisticRegression,0.7344,0.4511,0.7448,0.5619,0.8144,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline logistic regression with class_weight...
6,2026-04-28_225754_logistic_regression_baseline_v1,logistic_regression_baseline_v1,LogisticRegression,0.7703,0.4981,0.6403,0.5603,0.8162,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline logistic regression with class_weight...
8,2026-04-28_230732_logistic_regression_baseline_v2,logistic_regression_baseline_v2,LogisticRegression,0.7703,0.4981,0.6403,0.5603,0.8162,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline logistic regression using engineered ...
7,2026-04-28_230240_logistic_regression_baseline_v1,logistic_regression_baseline_v1,LogisticRegression,0.7269,0.4430,0.7558,0.5586,0.8201,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline logistic regression with class_weight...
1,2026-04-27_140741_xgboost_baseline_v1,xgboost_baseline_v1,XGBClassifier,0.8069,0.6103,0.4301,0.5046,0.8200,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline XGBoost model before hyperparameter t...
3,2026-04-27_140758_xgboost_baseline_v1,xgboost_baseline_v1,XGBClassifier,0.8069,0.6103,0.4301,0.5046,0.8200,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline XGBoost model before hyperparameter t...
5,2026-04-28_061724_xgboost_baseline_v1,xgboost_baseline_v1,XGBClassifier,0.8069,0.6103,0.4301,0.5046,0.8200,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline XGBoost model before hyperparameter t...
9,2026-04-28_230734_xgboost_baseline_v2,xgboost_baseline_v2,XGBClassifier,0.8078,0.6180,0.4168,0.4978,0.8179,/Users/eugene/Desktop/Emory/Projects/subrogati...,"Baseline XGBoost using engineered features, en..."


## 8. Recommended model selection logic

For this project, accuracy should not be the main metric for choosing the best model. In a subrogation problem, the target class is likely imbalanced, meaning there may be far fewer claims with subrogation potential than claims without it. Because of that, a model can look accurate while still missing many of the cases we actually care about.

The model should be selected based on the business goal:

Use recall when the priority is to identify as many potential recovery opportunities as possible.
Use precision when false positives would create too much unnecessary manual review work.
Use F1-score when we want a reasonable balance between precision and recall.
Use ROC-AUC to understand how well the model separates likely subrogation claims from non-subrogation claims overall.

For the baseline phase, F1-score is a useful headline metric because it balances precision and recall. However, precision and recall should still be reviewed together before deciding which model is actually better.

In [20]:
best_models = metrics_df[comparison_cols].sort_values(
    by="metrics.f1_score",
    ascending=False
)

best_models.head(5)


,run_id,model_name,model_family,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,model_path,notes
0,2026-04-27_140727_logistic_regression_baseline_v1,logistic_regression_baseline_v1,LogisticRegression,0.7344,0.4511,0.7448,0.5619,0.8144,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline logistic regression with class_weight...
2,2026-04-27_140757_logistic_regression_baseline_v1,logistic_regression_baseline_v1,LogisticRegression,0.7344,0.4511,0.7448,0.5619,0.8144,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline logistic regression with class_weight...
4,2026-04-28_061722_logistic_regression_baseline_v1,logistic_regression_baseline_v1,LogisticRegression,0.7344,0.4511,0.7448,0.5619,0.8144,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline logistic regression with class_weight...
6,2026-04-28_225754_logistic_regression_baseline_v1,logistic_regression_baseline_v1,LogisticRegression,0.7703,0.4981,0.6403,0.5603,0.8162,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline logistic regression with class_weight...
8,2026-04-28_230732_logistic_regression_baseline_v2,logistic_regression_baseline_v2,LogisticRegression,0.7703,0.4981,0.6403,0.5603,0.8162,/Users/eugene/Desktop/Emory/Projects/subrogati...,Baseline logistic regression using engineered ...
